# EPI: retreino rápido do YOLO26s
Mantém o split por cena original. Primeiro tente recuperar os pesos na aba Output da versão anterior do Kaggle. Se não estiverem disponíveis, execute este notebook com GPU.


Execute em sessão interativa para tentar downloads automáticos. O Kaggle pode bloquear JavaScript; nesse caso baixe pelo painel Output enquanto a sessão estiver ativa. Para persistir os arquivos, use Save Version (Save & Run All) e confira o Output da versão concluída; o modo em segundo plano não dispara downloads no navegador. O primeiro ZIP é provisório; use o ZIP final com métricas e hash.

## 1. Setup

In [ ]:
%pip -q install ultralytics==8.4.215 imagehash


In [ ]:
import os, shutil, random, yaml
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import imagehash
from PIL import Image
from IPython.display import display
from scipy.sparse.csgraph import connected_components
from ultralytics import YOLO

os.environ['WANDB_MODE'] = 'disabled'
BASE = Path('/kaggle/working')
RUNS = BASE / 'runs'
NAMES = ['helmet', 'no_helmet', 'no_vest', 'person', 'vest']
SPLITS = ['train', 'valid', 'test']
DS = next(Path('/kaggle/input').rglob('data.yaml')).parent
print(DS)
if not torch.cuda.is_available():
    raise RuntimeError('Ative uma GPU nas configurações do Kaggle antes de continuar.')
print(torch.cuda.get_device_name(0))

## 2. Hash perceptual de todas as imagens

In [ ]:
itens = [(s, p) for s in SPLITS for p in sorted((DS / s / 'images').glob('*'))]
H = np.array([imagehash.phash(Image.open(p).convert('RGB'), hash_size=8).hash.flatten()
              for _, p in itens], dtype=np.float32)
soma = H.sum(1)
D = (soma[:, None] + soma[None, :] - 2 * H @ H.T).astype(np.int16)
print(len(itens), 'imagens')

## 3. Grupos por cena
Imagens com distância de hash menor ou igual a `LIMIAR` ficam no mesmo grupo (componentes conectados).

In [ ]:
LIMIAR = 10
_, grupo = connected_components(D <= LIMIAR, directed=False)
tam = pd.Series(grupo).value_counts()
print('grupos:', len(tam))
print('maior grupo:', tam.iloc[0], 'imagens')
print('grupos com 1 imagem:', int((tam == 1).sum()))
print(tam.head(10).to_string())
tam.clip(upper=30).hist(bins=30, figsize=(7, 3))
plt.title('Imagens por grupo (cortado em 30)')
plt.show()

## 4. Novo split por grupo
Grupos maiores que 10% do dataset vão para o treino. Confira a tabela de classes por split antes de seguir.

In [ ]:
rng = random.Random(42)
ids = list(tam.index)
rng.shuffle(ids)

alvo = 0.1 * len(itens)
cont = {'train': 0, 'valid': 0, 'test': 0}
destino = {}
for g in ids:
    n = int(tam[g])
    if n > alvo:
        d = 'train'
    elif cont['test'] < alvo:
        d = 'test'
    elif cont['valid'] < alvo:
        d = 'valid'
    else:
        d = 'train'
    destino[g] = d
    cont[d] += n

def classes_da_imagem(s, p):
    lab = DS / s / 'labels' / (p.stem + '.txt')
    if not lab.exists():
        return []
    return [int(l.split()[0]) for l in lab.read_text().splitlines() if l.strip()]

meta = pd.DataFrame({
    'arquivo': [f'{s}_{p.stem}{p.suffix}' for s, p in itens],
    'split_original': [s for s, _ in itens],
    'grupo': grupo,
})
meta['split'] = meta.grupo.map(destino)
meta['classes'] = [classes_da_imagem(s, p) for s, p in itens]

display(meta.split.value_counts().to_frame('imagens'))
display(meta.groupby('split').grupo.nunique().to_frame('grupos'))

inst = meta[['split', 'classes']].explode('classes').dropna()
tab = pd.crosstab(inst.classes.map(dict(enumerate(NAMES))), inst.split)[SPLITS]
display(tab)

In [ ]:
ix = {s: np.where(meta.split == s)[0] for s in SPLITS}
for a, b in [('test', 'train'), ('valid', 'train'), ('test', 'valid')]:
    d = D[np.ix_(ix[a], ix[b])].min(1)
    print(a, 'vs', b, '| distância mínima:', d.min(), '| imagens com vizinho <= 10:', int((d <= 10).sum()))

## 5. Montar o dataset

In [ ]:
OUT = Path('/kaggle/temp/ppe_grupos')
shutil.rmtree(OUT, ignore_errors=True)
for s in SPLITS:
    (OUT / s / 'images').mkdir(parents=True)
    (OUT / s / 'labels').mkdir(parents=True)

for (s0, p), s, arq in zip(itens, meta.split, meta.arquivo):
    shutil.copy(p, OUT / s / 'images' / arq)
    lab = DS / s0 / 'labels' / (p.stem + '.txt')
    texto = lab.read_text() if lab.exists() else ''
    (OUT / s / 'labels' / (Path(arq).stem + '.txt')).write_text(texto)

yaml.safe_dump(
    {'path': str(OUT), 'train': 'train/images', 'val': 'valid/images', 'test': 'test/images',
     'nc': len(NAMES), 'names': NAMES},
    open(OUT / 'data.yaml', 'w'))

DATA = str(OUT / 'data.yaml')
print({s: len(list((OUT / s / 'images').glob('*'))) for s in SPLITS})

## 6. Treino apenas do YOLO26s
40 épocas por padrão, parada antecipada com paciência 10, cache em disco e AMP. Para reproduzir a configuração original de épocas, use 80 e paciência 20 (cache e versão fixada mantidos). O novo treino precisa de avaliação própria.


In [ ]:
EPOCHS = 40
PATIENCE = 10
E = BASE / 'entrega_grupo'
E.mkdir(exist_ok=True)

# Exportação durante o treino, sem depender do FileLink do Kaggle.
import hashlib, base64, json
from IPython.display import Javascript, display

PESO_FINAL = E / 'best_epi_grupo_yolo26s.pt'
ZIP_FINAL = BASE / 'epi_resultados_grupo.zip'
shutil.copy2(OUT / 'data.yaml', E / 'data_epi_grupo.yaml')
meta.drop(columns='classes').to_csv(E / 'split_por_grupo.csv', index=False)


def baixar_agora(arquivo):
    # Envia os bytes pela saída da célula; não usa URL do proxy do Kaggle.
    dados = base64.b64encode(Path(arquivo).read_bytes()).decode('ascii')
    nome = json.dumps(Path(arquivo).name)
    display(Javascript("""(function(){
      const raw = atob('%s');
      const bytes = new Uint8Array(raw.length);
      for(let i=0;i<raw.length;i++) bytes[i]=raw.charCodeAt(i);
      const url=URL.createObjectURL(new Blob([bytes],{type:'application/octet-stream'}));
      const a=document.createElement('a'); a.href=url; a.download=%s;
      document.body.appendChild(a); a.click(); a.remove();
      setTimeout(()=>URL.revokeObjectURL(url),60000);
    })();""" % (dados, nome)))


def salvar_peso(trainer):
    origem = Path(trainer.best)
    if not origem.is_file():
        return
    shutil.copy2(origem, PESO_FINAL)
    (E / 'sha256.txt').write_text(hashlib.sha256(PESO_FINAL.read_bytes()).hexdigest() + '  ' + PESO_FINAL.name + '\n')
    shutil.make_archive(str(ZIP_FINAL.with_suffix('')), 'zip', E)
    print('Backup atualizado:', ZIP_FINAL, flush=True)
    # Uma tentativa na primeira época, sem baixar dezenas de arquivos.
    if trainer.epoch == 0:
        baixar_agora(ZIP_FINAL)

# Para recuperar um checkpoint de uma execução anterior, informe seu caminho.
CHECKPOINT = None  # ex.: '/kaggle/input/.../last.pt'
if CHECKPOINT:
    modelo = YOLO(CHECKPOINT)
    modelo.add_callback('on_model_save', salvar_peso)
    modelo.train(resume=True, data=DATA, device=0)
else:
    modelo = YOLO('yolo26s.pt')
    modelo.add_callback('on_model_save', salvar_peso)
    modelo.train(data=DATA, epochs=EPOCHS, imgsz=640, batch=16,
                 patience=PATIENCE, seed=42, device=0, amp=True,
                 cache='disk', workers=4, save=True, save_period=10,
                 project=str(RUNS), name='grupo_yolo26s', exist_ok=False)

peso = Path(modelo.trainer.best)
assert peso.is_file(), f'Peso não encontrado: {peso}'
PESO_FINAL = E / 'best_epi_grupo_yolo26s.pt'
shutil.copy2(peso, PESO_FINAL)
shutil.copy2(OUT / 'data.yaml', E / 'data_epi_grupo.yaml')
meta.drop(columns='classes').to_csv(E / 'split_por_grupo.csv', index=False)
print('Peso salvo:', PESO_FINAL)
shutil.make_archive(str(ZIP_FINAL.with_suffix('')), 'zip', E)
print('ZIP do melhor modelo:', ZIP_FINAL)
baixar_agora(ZIP_FINAL)


## 7. Avaliação do modelo salvo
Uma avaliação por split, reutilizando o resultado de teste para as métricas por classe.


In [ ]:
resultados = []
for split in ['val', 'test']:
    r = YOLO(str(PESO_FINAL)).val(data=DATA, split=split, imgsz=640,
                                  batch=16, device=0, plots=(split == 'test'),
                                  project=str(RUNS), name='avaliacao_' + split, exist_ok=False)
    resultados.append({'modelo': 'yolo26s', 'split': split, 'precisao': r.box.mp,
                       'recall': r.box.mr, 'mAP50': r.box.map50,
                       'mAP50-95': r.box.map, 'ms_por_imagem': r.speed['inference']})
    if split == 'test':
        r_s = r
        pc_s = pd.DataFrame({'classe': [r.names[i] for i in r.box.ap_class_index],
                             'precisao': r.box.p, 'recall': r.box.r,
                             'mAP50': r.box.ap50, 'mAP50-95': r.box.ap})
tabela = pd.DataFrame(resultados)
tabela.to_csv(E / 'metricas_grupo.csv', index=False)
pc_s.to_csv(E / 'metricas_por_classe_teste_yolo26s.csv', index=False)
display(tabela.round(4))
display(pc_s.round(4))


In [ ]:
for nome in ['confusion_matrix_normalized.png', 'confusion_matrix.png']:
    f = Path(r_s.save_dir) / nome
    if f.exists():
        plt.figure(figsize=(8, 7))
        plt.imshow(plt.imread(f))
        plt.axis('off')
        plt.show()
        break

## 8. Salvar artefatos

In [ ]:
import hashlib
sha = hashlib.sha256(PESO_FINAL.read_bytes()).hexdigest()
(E / 'sha256.txt').write_text(f'{sha}  {PESO_FINAL.name}\n')
shutil.make_archive(str(BASE / 'epi_resultados_grupo'), 'zip', E)
print('SHA-256:', sha)
print(sorted(p.name for p in E.iterdir()))
baixar_agora(BASE / 'epi_resultados_grupo.zip')


In [ ]:
# Execute somente esta célula para tentar baixar novamente, sem retreinar.
assert ZIP_FINAL.is_file(), 'ZIP não encontrado na sessão atual.'
baixar_agora(ZIP_FINAL)